In [1]:
#import
from torch import nn,optim
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import numpy as np
from matplotlib import pyplot as plt
import os
import copy
import math
import sys
import importlib

In [2]:
# 코랩 환경인지 확인하는 조건문
if 'google.colab' in sys.modules:
    print("현재 환경: Google Colab")
    # 코랩 전용 설정 (예: 드라이브 마운트)
    from google.colab import drive
    drive.mount('/content/drive')
    path='/content/drive/MyDrive/02_학업/02_연구 및 프로젝트/2512-2602_Dash 연구인턴/pytorch_practice'
    # path='/content/drive/MyDrive//pytorch_practice'
    sys.path.append(path)
else:
    print("현재 환경: Local Jupyter")
    # 로컬 전용 설정 (예: 경로 설정)
    # path = './'
    path = r'g:\내 드라이브\02_학업\02_연구 및 프로젝트\2512-2602_Dash 연구인턴\pytorch_practice'
# from my_module import *
import my_module2 as mm
print(f"작업 경로: {path}")

현재 환경: Local Jupyter
작업 경로: g:\내 드라이브\02_학업\02_연구 및 프로젝트\2512-2602_Dash 연구인턴\pytorch_practice


In [ ]:
importlib.reload(mm)
import my_module2 as mm

In [3]:
model_dir = os.path.join(path, 'download')
print(os.listdir(model_dir))

root=os.path.join(path, 'data','test')
os.makedirs(root, exist_ok=True)
print(os.listdir(root))

checkpoint_dir = os.path.join(path, 'checkpoints')
print(os.listdir(checkpoint_dir))

DEVICE= 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'current device: {DEVICE}')

['cifar10_vgg16_bn-6ee7ea24.pt']
['cifar-10-batches-py', 'cifar-10-python.tar.gz']
['best_model.pt', 'CNN_CIFAR10_final_weights.pth', 'ckpt_ep1.pt', 'ckpt_ep2.pt', 'ckpt_ep3.pt', 'ckpt_ep5.pt', 'ckpt_ep10.pt', 'ckpt_ep15.pt', 'ckpt_ep20.pt', 'saved_results']
current device: cpu


In [4]:
model = torch.hub.load("chenyaofo/pytorch-cifar-models", "cifar10_vgg16_bn", pretrained=True).to(DEVICE)

Using cache found in C:\Users\darwin5991/.cache\torch\hub\chenyaofo_pytorch-cifar-models_master


In [5]:
BATCH_SIZE=128
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

full_train_DS = datasets.CIFAR10(root=root, train=True, download=True, transform=transform_train)
test_DS = datasets.CIFAR10(root=root, train=False, download=True, transform=transform_test)

train_size = 45000
val_size = 5000
train_DS, val_DS = random_split(full_train_DS, [train_size, val_size])

train_DL=torch.utils.data.DataLoader(train_DS, batch_size=BATCH_SIZE, shuffle=True)
val_DL = torch.utils.data.DataLoader(val_DS, batch_size=BATCH_SIZE, shuffle=False) # 검증용 추가
test_DL=torch.utils.data.DataLoader(test_DS, batch_size=BATCH_SIZE, shuffle=False)

print(f"Data loaded: Train({len(train_DS)}), Val({len(val_DS)}), Test({len(test_DS)})")

Files already downloaded and verified
Files already downloaded and verified
Data loaded: Train(45000), Val(5000), Test(10000)


In [166]:
print(len(train_DL))

352


In [149]:
print(train_DS.dataset.classes)

['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


In [7]:
rcorrect,acc=mm.Test(model, test_DL, DEVICE)
print(f"Test accuracy: {rcorrect}/{len(test_DL.dataset)} ({acc} %)")

Test accuracy: 9416/10000 (94.2 %)


In [ ]:
print(model.parameters)

<bound method Module.parameters of VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU(inplace=True)
    (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (7): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (9): ReLU(inplace=True)
    (10): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (12): ReLU(inplace=True)
    (13): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mo

In [164]:
weight_param_list = [p for p in model.named_parameters() if 'weight' in p[0]and p[1].dim() ==4]
params_dict = dict(model.named_parameters())

for name, param in weight_param_list:
    print(f"{name}: {params_dict[name].shape}")

weight_param_list = [p for p in model.named_parameters() if 'weight' in p[0]and p[1].dim() ==2]
for name, param in weight_param_list:
    print(f"{name}: {params_dict[name].shape}")


features.0.weight: torch.Size([64, 3, 3, 3])
features.3.weight: torch.Size([64, 64, 3, 3])
features.7.weight: torch.Size([128, 64, 3, 3])
features.10.weight: torch.Size([128, 128, 3, 3])
features.14.weight: torch.Size([256, 128, 3, 3])
features.17.weight: torch.Size([256, 256, 3, 3])
features.20.weight: torch.Size([256, 256, 3, 3])
features.24.weight: torch.Size([512, 256, 3, 3])
features.27.weight: torch.Size([512, 512, 3, 3])
features.30.weight: torch.Size([512, 512, 3, 3])
features.34.weight: torch.Size([512, 512, 3, 3])
features.37.weight: torch.Size([512, 512, 3, 3])
features.40.weight: torch.Size([512, 512, 3, 3])
classifier.0.weight: torch.Size([512, 512])
classifier.3.weight: torch.Size([512, 512])
classifier.6.weight: torch.Size([10, 512])


In [153]:
weight_param_list = [p for p in model.named_parameters() if 'weight' in p[0]and p[1].dim() ==4]
print(len(weight_param_list))
print(weight_param_list[0][1].shape)
print(weight_param_list[1][1].shape)

weight_param_list = [p for p in model.named_parameters() if 'weight' in p[0]and p[1].dim() ==2]
print(len(weight_param_list))

# name,weight=weight_param_list[0]
# print(name)
# print(weight.shape)


# weight_tensor=weight.data.clone()


13
torch.Size([64, 3, 3, 3])
torch.Size([64, 64, 3, 3])
3


In [ ]:
import sklearn
import sklearn as sk

from sklearn.cluster import KMeans

In [ ]:
X = weight_tensor.cpu().numpy().reshape(-1,1)
# print(X.shape())
n=int(X.shape[0]//8)
print(n)


216


In [ ]:
kmeans = KMeans(n_clusters=32, random_state=0).fit(X)


c:\Users\darwin5991\.conda\envs\phh\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=7.
  warnings.warn(


In [ ]:
labels = kmeans.labels_                 # 각 필터의 클러스터
centers = kmeans.cluster_centers_       # 각 클러스터의 중심 (벡터)
# 클러스터 중심으로 가중치 재구성 (예: 양자화)
new_weights = centers[labels].reshape(weight_tensor.shape)  # numpy -> 원래 shape
new_weight_tensor = torch.from_numpy(new_weights).to(weight_tensor.device).type_as(weight_tensor)

In [ ]:
pruned_model = copy.deepcopy(model)
params_dict = dict(pruned_model.named_parameters())
with torch.no_grad():
    params_dict[name].copy_(new_weight_tensor)



In [ ]:
rcorrect,acc=mm.Test(pruned_model, test_DL, DEVICE)
print(f"Test accuracy: {rcorrect}/{len(test_DL.dataset)} ({acc} %)")

Test accuracy: 9413/10000 (94.1 %)


In [ ]:
weight_param_list = [p for p in model.named_parameters() if 'weight' in p[0]and p[1].dim() ==4]
print(len(weight_param_list))

pruned_model = copy.deepcopy(model)
params_dict = dict(pruned_model.named_parameters())


for l in range (len(weight_param_list)):
    name,weight=weight_param_list[l]
    # print(name)
    # print(weight.shape)
    weight_tensor=weight.data.clone()

    X = weight_tensor.cpu().numpy().reshape(-1,1)
    kmeans = KMeans(n_clusters=32, random_state=0).fit(X)

    labels = kmeans.labels_
    centers = kmeans.cluster_centers_

    new_weights = centers[labels].reshape(weight_tensor.shape)
    new_weight_tensor = torch.from_numpy(new_weights).to(DEVICE).type_as(weight_tensor)

    with torch.no_grad():
        params_dict[name].copy_(new_weight_tensor)


rcorrect,acc=mm.Test(pruned_model, test_DL, DEVICE)
print(f"Test accuracy: {rcorrect}/{len(test_DL.dataset)} ({acc} %)")


13
features.0.weight
features.3.weight


c:\Users\darwin5991\.conda\envs\phh\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=7.
  warnings.warn(


features.7.weight
features.10.weight
features.14.weight
features.17.weight
features.20.weight
features.24.weight
features.27.weight
features.30.weight
features.34.weight
features.37.weight
features.40.weight
Test accuracy: 9394/10000 (93.9 %)


In [ ]:
weight_param_list = [p for p in model.named_parameters() if 'weight' in p[0]and p[1].dim() ==4]
print(len(weight_param_list))

pruned_model = copy.deepcopy(model)
params_dict = dict(pruned_model.named_parameters())


for l in range (len(weight_param_list)):
    name,weight=weight_param_list[l]
    print(name)
    # print(weight.shape)
    weight_tensor=weight.data.clone()

    X = weight_tensor.cpu().numpy().reshape(-1,1)
    kmeans = KMeans(n_clusters=16, random_state=0).fit(X)

    labels = kmeans.labels_
    centers = kmeans.cluster_centers_

    new_weights = centers[labels].reshape(weight_tensor.shape)
    new_weight_tensor = torch.from_numpy(new_weights).to(DEVICE).type_as(weight_tensor)

    with torch.no_grad():
        params_dict[name].copy_(new_weight_tensor)


rcorrect,acc=mm.Test(pruned_model, test_DL, DEVICE)
print(f"Test accuracy: {rcorrect}/{len(test_DL.dataset)} ({acc} %)")




# weight_tensor=weight.data.clone()

13
features.0.weight
features.3.weight
features.7.weight


c:\Users\darwin5991\.conda\envs\phh\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=7.
  warnings.warn(


features.10.weight
features.14.weight
features.17.weight
features.20.weight
features.24.weight
features.27.weight
features.30.weight
features.34.weight
features.37.weight
features.40.weight
Test accuracy: 9341/10000 (93.4 %)


In [ ]:
weight_param_list = [p for p in model.named_parameters() if 'weight' in p[0]and p[1].dim() ==4]
print(len(weight_param_list))

pruned_model = copy.deepcopy(model)
params_dict = dict(pruned_model.named_parameters())


for l in range (len(weight_param_list)):
    name,weight=weight_param_list[l]
    print(name)
    # print(weight.shape)
    weight_tensor=weight.data.clone()

    X = weight_tensor.cpu().numpy().reshape(-1,1)
    kmeans = KMeans(n_clusters=12, random_state=0).fit(X)

    labels = kmeans.labels_
    centers = kmeans.cluster_centers_

    new_weights = centers[labels].reshape(weight_tensor.shape)
    new_weight_tensor = torch.from_numpy(new_weights).to(DEVICE).type_as(weight_tensor)

    with torch.no_grad():
        params_dict[name].copy_(new_weight_tensor)


rcorrect,acc=mm.Test(pruned_model, test_DL, DEVICE)
print(f"Test accuracy: {rcorrect}/{len(test_DL.dataset)} ({acc} %)")




# weight_tensor=weight.data.clone()

13
features.0.weight
features.3.weight
features.7.weight


c:\Users\darwin5991\.conda\envs\phh\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=7.
  warnings.warn(


features.10.weight
features.14.weight
features.17.weight
features.20.weight
features.24.weight
features.27.weight
features.30.weight
features.34.weight
features.37.weight
features.40.weight
Test accuracy: 9267/10000 (92.7 %)


In [ ]:
weight_param_list = [p for p in model.named_parameters() if 'weight' in p[0]and p[1].dim() ==4]
print(len(weight_param_list))

pruned_model = copy.deepcopy(model)
params_dict = dict(pruned_model.named_parameters())


for l in range (len(weight_param_list)):
    name,weight=weight_param_list[l]
    print(name)
    # print(weight.shape)
    weight_tensor=weight.data.clone()

    X = weight_tensor.cpu().numpy().reshape(-1,1)
    kmeans = KMeans(n_clusters=8, random_state=0).fit(X)

    labels = kmeans.labels_
    centers = kmeans.cluster_centers_

    new_weights = centers[labels].reshape(weight_tensor.shape)
    new_weight_tensor = torch.from_numpy(new_weights).to(DEVICE).type_as(weight_tensor)

    with torch.no_grad():
        params_dict[name].copy_(new_weight_tensor)


rcorrect,acc=mm.Test(pruned_model, test_DL, DEVICE)
print(f"Test accuracy: {rcorrect}/{len(test_DL.dataset)} ({acc} %)")




# weight_tensor=weight.data.clone()

13
features.0.weight
features.3.weight
features.7.weight


c:\Users\darwin5991\.conda\envs\phh\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=7.
  warnings.warn(


features.10.weight
features.14.weight
features.17.weight
features.20.weight
features.24.weight
features.27.weight
features.30.weight
features.34.weight
features.37.weight
features.40.weight
Test accuracy: 8938/10000 (89.4 %)


In [ ]:
weight_param_list = [p for p in model.named_parameters() if 'weight' in p[0]and p[1].dim() ==4]
print(len(weight_param_list))

quanted_model = copy.deepcopy(model)
params_dict = dict(quanted_model.named_parameters())

name,weight=weight_param_list[0]
print(name)
print(weight.shape)

weight_tensor=weight.data.clone()






13
features.0.weight
torch.Size([64, 3, 3, 3])


In [ ]:
print(torch.max(weight_tensor))
print(torch.min(weight_tensor))
r_max=torch.max(weight_tensor)
r_min=torch.min(weight_tensor)



tensor(0.4982)
tensor(-0.6226)


bit width=n
q_min=-2^(n-1)
q_max=2^n -1

In [10]:
bit_width = list(range(2, 9))
q_min = [ -2**(b-1) for b in bit_width ]
q_max = [  2**(b-1) - 1   for b in bit_width ]

In [ ]:
print(bit_width)
print(q_max)
print(q_min)

[2, 3, 4, 5, 6, 7, 8]
[1, 3, 7, 15, 31, 63, 127]
[-2, -4, -8, -16, -32, -64, -128]


In [ ]:
print(r_max-r_min)
print(q_max[1]-q_min[1])

print((r_max-r_min)/7)

tensor(1.1208)
7
tensor(0.1601)


In [ ]:
S= [(r_max - r_min) / (q_max - q_min) for q_min, q_max in zip (q_min, q_max)]
print(S)

[tensor(0.3736), tensor(0.1601), tensor(0.0747), tensor(0.0362), tensor(0.0178), tensor(0.0088), tensor(0.0044)]


In [ ]:
Z=[torch.round(q_min - r_min/S) for S, q_min in zip (S, q_min)]
print(Z)

[tensor(-0.), tensor(-0.), tensor(0.), tensor(1.), tensor(3.), tensor(7.), tensor(14.)]


In [ ]:
bit_num=3
idx=bit_num-2

quanted_weight=torch.round(weight_tensor/S[idx]+Z[idx])

In [ ]:
print(quanted_weight[3][1][:])

tensor([[-0., -0., -0.],
        [-0., 0., 0.],
        [0., 1., 0.]])


In [ ]:
print(((quanted_weight-Z[idx])*S[idx])[3][1][:])


tensor([[0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000],
        [0.0000, 0.1601, 0.0000]])


In [ ]:
print(r_max)
print(torch.max((quanted_weight-Z[idx])*S[idx]))

print(r_min)
print(torch.min((quanted_weight-Z[idx])*S[idx]))

tensor(0.4982)
tensor(0.4804)
tensor(-0.6226)
tensor(-0.6405)


In [ ]:
weight_dequanted=(quanted_weight-Z[idx])*S[idx]

In [ ]:
print(torch.sum(weight_tensor-weight_dequanted))

tensor(3.8484)


In [ ]:
for i in range(len(bit_width)):
    print((torch.sum(torch.abs(weight_tensor-(torch.round(weight_tensor/S[i]+Z[i])-Z[i])*S[i])))/weight_tensor.numel())

tensor(0.0577)
tensor(0.0346)
tensor(0.0169)
tensor(0.0085)
tensor(0.0044)
tensor(0.0022)
tensor(0.0011)


In [9]:
weight_param_list = [p for p in model.named_parameters() if 'weight' in p[0]and p[1].dim() ==4]
print(len(weight_param_list))

for bit_num in range(2, 9):
    idx=bit_num-2
    q_max_=q_max[idx]
    q_min_=q_min[idx]

    quanted_model = copy.deepcopy(model)
    params_dict = dict(quanted_model.named_parameters())

    for l in range (len(weight_param_list)):

        name,weight=weight_param_list[l]
        # print(name)
        # print(weight.shape)

        weight_tensor=weight.data.clone()

        r_max=torch.max(weight_tensor)
        r_min=torch.min(weight_tensor)
        S_= (r_max - r_min) / (q_max_ - q_min_)
        Z_=torch.round(q_min_ - r_min/S_)

        quanted_weight=torch.round(weight_tensor/S_+Z_)
        weight_dequanted=(quanted_weight-Z_)*S_

        with torch.no_grad():
            params_dict[name].copy_(weight_dequanted)

    rcorrect,acc=mm.Test(quanted_model, test_DL, DEVICE)
    print(f"Test accuracy({bit_num} 비트): {rcorrect}/{len(test_DL.dataset)} ({acc} %)")


13
Test accuracy(2 비트): 1000/10000 (10.0 %)
Test accuracy(3 비트): 3948/10000 (39.5 %)
Test accuracy(4 비트): 9328/10000 (93.3 %)
Test accuracy(5 비트): 9364/10000 (93.6 %)
Test accuracy(6 비트): 9401/10000 (94.0 %)
Test accuracy(7 비트): 9410/10000 (94.1 %)
Test accuracy(8 비트): 9418/10000 (94.2 %)


In [10]:
weight_param_list = [p for p in model.named_parameters() if 'weight' in p[0]and p[1].dim() ==4]
print(len(weight_param_list))

for bit_num in range(2, 9):
    idx=bit_num-2
    q_max_=q_max[idx]
    q_min_=q_min[idx]

    quanted_model = copy.deepcopy(model)
    params_dict = dict(quanted_model.named_parameters())

    for l in range (len(weight_param_list)):

        name,weight=weight_param_list[l]
        # print(name)
        # print(weight.shape)

        weight_tensor=weight.data.clone()
        quanted_weight=torch.zeros_like(weight_tensor)
        weight_dequanted=torch.zeros_like(weight_tensor)

        for k in range(weight_tensor.shape[0]):
            r_max=torch.max(weight_tensor[k])
            r_min=torch.min(weight_tensor[k])
            S_= (r_max - r_min) / (q_max_ - q_min_)
            Z_=torch.round(q_min_ - r_min/S_)

            quanted_weight[k]=torch.round(weight_tensor[k]/S_+Z_)
            weight_dequanted[k]=(quanted_weight[k]-Z_)*S_

        with torch.no_grad():
            params_dict[name].copy_(weight_dequanted)

    rcorrect,acc=mm.Test(quanted_model, test_DL, DEVICE)
    print(f"Test accuracy({bit_num} 비트): {rcorrect}/{len(test_DL.dataset)} ({acc} %)")

13
Test accuracy(2 비트): 1931/10000 (19.3 %)
Test accuracy(3 비트): 9116/10000 (91.2 %)
Test accuracy(4 비트): 9370/10000 (93.7 %)
Test accuracy(5 비트): 9413/10000 (94.1 %)
Test accuracy(6 비트): 9413/10000 (94.1 %)
Test accuracy(7 비트): 9416/10000 (94.2 %)
Test accuracy(8 비트): 9414/10000 (94.1 %)


In [11]:
weight_param_list = [p for p in model.named_parameters() if 'weight' in p[0]and p[1].dim() >1]
print(len(weight_param_list))

for bit_num in range(2, 9):
    idx=bit_num-2
    q_max_=q_max[idx]
    q_min_=q_min[idx]

    quanted_model = copy.deepcopy(model)
    params_dict = dict(quanted_model.named_parameters())

    for l in range (len(weight_param_list)):

        name,weight=weight_param_list[l]
        # print(name)
        # print(weight.shape)

        weight_tensor=weight.data.clone()
        quanted_weight=torch.zeros_like(weight_tensor)
        weight_dequanted=torch.zeros_like(weight_tensor)

        for k in range(weight_tensor.shape[0]):
            r_max=torch.max(weight_tensor[k])
            r_min=torch.min(weight_tensor[k])
            S_= (r_max - r_min) / (q_max_ - q_min_)
            Z_=torch.round(q_min_ - r_min/S_)

            quanted_weight[k]=torch.round(weight_tensor[k]/S_+Z_)
            weight_dequanted[k]=(quanted_weight[k]-Z_)*S_

        with torch.no_grad():
            params_dict[name].copy_(weight_dequanted)

    rcorrect,acc=mm.Test(quanted_model, test_DL, DEVICE)
    print(f"Test accuracy({bit_num} 비트): {rcorrect}/{len(test_DL.dataset)} ({acc} %)")








16
Test accuracy(2 비트): 1698/10000 (17.0 %)
Test accuracy(3 비트): 9114/10000 (91.1 %)
Test accuracy(4 비트): 9374/10000 (93.7 %)
Test accuracy(5 비트): 9415/10000 (94.2 %)
Test accuracy(6 비트): 9413/10000 (94.1 %)
Test accuracy(7 비트): 9416/10000 (94.2 %)
Test accuracy(8 비트): 9414/10000 (94.1 %)


In [11]:
bit_width = list(range(2, 9))
q_min = [ -2**(b-1) for b in bit_width ]
q_max = [  2**(b-1) - 1   for b in bit_width ]

In [35]:
for name, module in model.named_children():
    print(name)

print(list(model._modules.keys()))


features
classifier
['features', 'classifier']


In [25]:
print(*list(model.features.children())[-10:], sep="\n")

Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
BatchNorm2d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
ReLU(inplace=True)
Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
BatchNorm2d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
ReLU(inplace=True)
Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
BatchNorm2d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
ReLU(inplace=True)
MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)


In [12]:
weight_param_list = [p for p in model.named_parameters() if 'weight' in p[0]and p[1].dim() >1]
print(len(weight_param_list))

bit_num=3
idx=bit_num-2
q_max_=q_max[idx]
q_min_=q_min[idx]

quanted_model = copy.deepcopy(model)
params_dict = dict(quanted_model.named_parameters())


name,weight=weight_param_list[0]



# print(name)
# print(weight.shape)

weight_tensor=weight.data.clone()
quanted_weight=torch.zeros_like(weight_tensor)
weight_dequanted=torch.zeros_like(weight_tensor)

# for k in range(weight_tensor.shape[0]):
#     r_max=torch.max(weight_tensor[k])
#     r_min=torch.min(weight_tensor[k])
#     S_= (r_max - r_min) / (q_max_ - q_min_)
#     Z_=torch.round(q_min_ - r_min/S_)

#     quanted_weight[k]=torch.round(weight_tensor[k]/S_+Z_)
#     weight_dequanted[k]=(quanted_weight[k]-Z_)*S_

# with torch.no_grad():
#     params_dict[name].copy_(weight_dequanted)

# rcorrect,acc=mm.Test(quanted_model, test_DL, DEVICE)
# print(f"Test accuracy({bit_num} 비트): {rcorrect}/{len(test_DL.dataset)} ({acc} %)")


16


In [39]:
print(*list(model.classifier.children()), sep="\n")

Linear(in_features=512, out_features=512, bias=True)
ReLU(inplace=True)
Dropout(p=0.5, inplace=False)
Linear(in_features=512, out_features=512, bias=True)
ReLU(inplace=True)
Dropout(p=0.5, inplace=False)
Linear(in_features=512, out_features=10, bias=True)


In [45]:
print(*list(model.classifier.children())[:3])
print(*list(model.classifier.children())[:4])
print(list(model.classifier.children())[3])

Linear(in_features=512, out_features=512, bias=True) ReLU(inplace=True) Dropout(p=0.5, inplace=False)
Linear(in_features=512, out_features=512, bias=True) ReLU(inplace=True) Dropout(p=0.5, inplace=False) Linear(in_features=512, out_features=512, bias=True)
Linear(in_features=512, out_features=512, bias=True)


In [59]:
# print(val_DL.dataset.dataset.data[0].shape)
# print(val_DL.dataset.dataset.targets[0])

print(len(val_DL.dataset.dataset.data))
print(len(val_DL.dataset.dataset.targets))

50000
50000


In [65]:
images, labels = next(iter(val_DL))
eg_input = images.to(DEVICE)

In [64]:
print(eg_input.shape)

torch.Size([3, 32, 32])


In [70]:
print(list(model.classifier.children())[3])

Linear(in_features=512, out_features=512, bias=True)


In [79]:
print(eg_input[:1][:].shape)

torch.Size([1, 3, 32, 32])


In [ ]:


input_sub=torch.nn.Sequential(*list(model.features.children()),torch.nn.Flatten(start_dim=1),*list(model.classifier.children())[:3]).to(DEVICE)

input=input_sub(eg_input)

output_sub=list(model.classifier.children())[3]

output=output_sub(input)


In [85]:
print(torch.max(input[1]))
print(input.amax(dim=1)[:5])
print(input.shape)
print(output.shape)

tensor(1.2126, grad_fn=<MaxBackward1>)
tensor([1.2193, 1.2126, 1.2298, 1.1788, 1.3790], grad_fn=<SliceBackward0>)
torch.Size([128, 512])
torch.Size([128, 512])


In [95]:
r_x_max=input.amax(dim=1)
r_x_min=input.amin(dim=1)
print(r_x_max.shape)
print(((r_x_max - r_x_min) / (q_max_ - q_min_)).shape)
S_x= (r_x_max - r_x_min) / (q_max_ - q_min_)
print(torch.round(q_min_ - r_x_min/S_x).shape)
Z_x=torch.round(q_min_ - r_x_min/S_x)
print(Z_x.mean().item())

torch.Size([128])
torch.Size([128])
torch.Size([128])
-4.0


In [98]:
images, labels = next(iter(val_DL))
eg_input = images.to(DEVICE)

input=input_sub(images.to(DEVICE))
output=output_sub(input)

r_x_max=input.amax(dim=1)
r_x_min=input.amin(dim=1)
S_x= (r_x_max - r_x_min) / (q_max_ - q_min_)
Z_x=torch.round(q_min_ - r_x_min/S_x)

r_y_max=output.amax(dim=1)
r_y_min=output.amin(dim=1)
S_y= (r_y_max - r_y_min) / (q_max_ - q_min_)
Z_y=torch.round(q_min_ - r_y_min/S_y)

print(S_x.mean().item())
print(torch.round(Z_x.mean()).item())
print(S_y.mean().item())
print(torch.round(Z_y.mean()).item())


0.1677388995885849
-4.0
0.21052658557891846
-3.0


In [9]:
input_sub=torch.nn.Sequential(*list(model.features.children()),torch.nn.Flatten(start_dim=1),*list(model.classifier.children())[:3]).to(DEVICE)
output_sub=list(model.classifier.children())[3]

In [7]:
bit_width = list(range(2, 9))
q_min = [ -2**(b-1) for b in bit_width ]
q_max = [  2**(b-1) - 1   for b in bit_width ]

In [101]:
bit_num=3
idx=bit_num-2
q_max_=q_max[idx]
q_min_=q_min[idx]

In [103]:
S_X_list=[]
S_Y_list=[]
Z_X_list=[]
Z_Y_list=[]

for images,_ in val_DL:
    input=input_sub(images.to(DEVICE))
    output=output_sub(input)

    r_x_max=input.amax(dim=1)
    r_x_min=input.amin(dim=1)
    S_x= (r_x_max - r_x_min) / (q_max_ - q_min_)
    Z_x=torch.round(q_min_ - r_x_min/S_x)

    r_y_max=output.amax(dim=1)
    r_y_min=output.amin(dim=1)
    S_y= (r_y_max - r_y_min) / (q_max_ - q_min_)
    Z_y=torch.round(q_min_ - r_y_min/S_y)
    

    S_X_list.append(S_x.mean().item())
    S_Y_list.append(S_y.mean().item())
    Z_X_list.append(Z_x.mean().item())
    Z_Y_list.append(Z_y.mean().item())


In [116]:
S_x=np.mean(S_X_list)
Z_x=np.mean(np.round(Z_X_list))
S_y=np.mean(S_Y_list)
Z_y=np.mean(np.round(Z_Y_list))

print(np.mean(S_X_list),S_x)
print(np.mean(np.round(Z_X_list)),Z_x)
print(np.mean(S_Y_list),S_y)
print(np.mean(np.round(Z_Y_list)),Z_y)

0.16983367018401624 0.16983367018401624
-4.0 -4.0
0.2144020102918148 0.2144020102918148
-3.0 -3.0


In [ ]:
# S_X, Z_X 구하기
# calibration set으로 S_X와 Z_X의 average 구함

# S_Y, Z_Y 구하기
# calibration set으로 S_Y와 Z_Y의 average 구함

In [117]:
bias=list(model.classifier.children())[3].bias
weight=list(model.classifier.children())[3].weight
print(bias.shape)
print(weight.shape)

torch.Size([512])
torch.Size([512, 512])


In [118]:
#Z_W=0 -> |r|_max 이용해서 symmetric quantization
#Z_b=0 -> 마찬가지로 symmetric
#S_b= S_W*S_x -> 실제로 비슷한가? 그렇게 설정해야하나? 차원이 같으니까 얼추 비슷할듯?

r_w_max=torch.max(torch.abs(weight))
r_b_max=torch.max(torch.abs(bias))
S_w= (r_w_max) / (q_max_ +1)
Z_w=0
S_b= S_w*S_x
Z_b=0

weight_tensor=weight.data.clone()
bias_tensor=bias.data.clone()
quanted_weight=torch.zeros_like(weight_tensor)
quanted_bias=torch.zeros_like(bias_tensor)
weight_dequanted=torch.zeros_like(weight_tensor)
bias_dequanted=torch.zeros_like(bias_tensor)


In [126]:
#q_bias=q_b-Z_x*q_w로 계산
quanted_weight=torch.round(weight_tensor/S_w+Z_w)
weight_dequanted=(quanted_weight-Z_w)*S_w
quanted_bias=torch.round(bias_tensor/S_b+Z_b)
bias_dequanted=(quanted_bias-Z_b)*S_b


q_bias=quanted_bias - Z_x*quanted_weight.sum(dim=1)



In [123]:
print(quanted_bias.shape)
print(quanted_weight.shape)
print(quanted_weight.sum(dim=1).shape)
print(Z_x)

torch.Size([512])
torch.Size([512, 512])
torch.Size([512])
-4.0


In [124]:
q_bias=quanted_bias - Z_x*quanted_weight.sum(dim=1)

In [125]:
print(q_bias.shape)

torch.Size([512])


In [131]:
print(S_w.item(),S_x,S_y)

print(quanted_weight.shape,q_bias.shape)
print(Z_y,Z_x)

print(input.shape)

0.014661702327430248 0.16983367018401624 0.2144020102918148
torch.Size([512, 512]) torch.Size([512])
-3.0 -4.0
torch.Size([8, 512])


In [136]:
print(len(test_DL))

79


In [ ]:
cnt=0
for images,_ in test_DL:
    input=input_sub(images.to(DEVICE))
    output=output_sub(input)
    cnt+=1
    q_x=torch.round(input/S_x+Z_x)
    # print(q_x.shape)
    # print(quanted_weight.shape)
    # print(q_bias.shape)
    q_y=torch.matmul(q_x,quanted_weight.T) + q_bias
    q_y=q_y/(S_w*S_x)*S_y + Z_y
    dequanted_y=(q_y - Z_y)*S_y

    # print(output.shape)
    # print(dequanted_y.shape)
    if cnt%10==0:
        print(cnt,torch.abs(output-dequanted_y).mean().item())

10 38.22188949584961
20 38.71548843383789
30 38.60688781738281
40 38.958553314208984
50 39.38074493408203
60 38.637184143066406
70 39.09743118286133


In [ ]:
bit_num=2
idx=bit_num-2
q_max_=q_max[idx]
q_min_=q_min[idx]

S_X_list=[]
S_Y_list=[]
Z_X_list=[]
Z_Y_list=[]

for images,_ in val_DL:
    input=input_sub(images.to(DEVICE))
    output=output_sub(input)

    r_x_max=input.amax(dim=1)
    r_x_min=input.amin(dim=1)
    S_x= (r_x_max - r_x_min) / (q_max_ - q_min_)
    Z_x=torch.round(q_min_ - r_x_min/S_x)

    r_y_max=output.amax(dim=1)
    r_y_min=output.amin(dim=1)
    S_y= (r_y_max - r_y_min) / (q_max_ - q_min_)
    Z_y=torch.round(q_min_ - r_y_min/S_y)
    

    S_X_list.append(S_x.mean().item())
    S_Y_list.append(S_y.mean().item())
    Z_X_list.append(Z_x.mean().item())
    Z_Y_list.append(Z_y.mean().item())

S_x=np.mean(S_X_list)
Z_x=np.mean(np.round(Z_X_list))
S_y=np.mean(S_Y_list)
Z_y=np.mean(np.round(Z_Y_list))

bias=list(model.classifier.children())[3].bias
weight=list(model.classifier.children())[3].weight

r_w_max=torch.max(torch.abs(weight))
r_b_max=torch.max(torch.abs(bias))
S_w= (r_w_max) / (q_max_ +1)
Z_w=0
S_b= S_w*S_x
Z_b=0

weight_tensor=weight.data.clone()
bias_tensor=bias.data.clone()
quanted_weight=torch.zeros_like(weight_tensor)
quanted_bias=torch.zeros_like(bias_tensor)
weight_dequanted=torch.zeros_like(weight_tensor)
bias_dequanted=torch.zeros_like(bias_tensor)

quanted_weight=torch.round(weight_tensor/S_w+Z_w)
weight_dequanted=(quanted_weight-Z_w)*S_w
quanted_bias=torch.round(bias_tensor/S_b+Z_b)
bias_dequanted=(quanted_bias-Z_b)*S_b


q_bias=quanted_bias - Z_x*quanted_weight.sum(dim=1)

sum=0
for images,_ in test_DL:
    input=input_sub(images.to(DEVICE))
    output=output_sub(input)

    q_x=torch.round(input/S_x+Z_x)
    q_y=torch.matmul(q_x,quanted_weight.T) + q_bias
    q_y=q_y*((S_w*S_x)/S_y) + Z_y
    
    dequanted_y=(q_y - Z_y)*S_y

    sum+=torch.abs(output-dequanted_y).mean().item()


print(f"Average error: {sum/len(test_DL):.4f}")

Average error: 0.1094


In [11]:
bit_num=5
idx=bit_num-2
q_max_=q_max[idx]
q_min_=q_min[idx]

S_X_list=[]
S_Y_list=[]
Z_X_list=[]
Z_Y_list=[]

for images,_ in val_DL:
    input=input_sub(images.to(DEVICE))
    output=output_sub(input)

    r_x_max=input.amax(dim=1)
    r_x_min=input.amin(dim=1)
    S_x= (r_x_max - r_x_min) / (q_max_ - q_min_)
    Z_x=torch.round(q_min_ - r_x_min/S_x)

    r_y_max=output.amax(dim=1)
    r_y_min=output.amin(dim=1)
    S_y= (r_y_max - r_y_min) / (q_max_ - q_min_)
    Z_y=torch.round(q_min_ - r_y_min/S_y)
    

    S_X_list.append(S_x.mean().item())
    S_Y_list.append(S_y.mean().item())
    Z_X_list.append(Z_x.mean().item())
    Z_Y_list.append(Z_y.mean().item())

S_x=np.mean(S_X_list)
Z_x=np.mean(np.round(Z_X_list))
S_y=np.mean(S_Y_list)
Z_y=np.mean(np.round(Z_Y_list))

bias=list(model.classifier.children())[3].bias
weight=list(model.classifier.children())[3].weight

r_w_max=torch.max(torch.abs(weight))
r_b_max=torch.max(torch.abs(bias))
S_w= (r_w_max) / (q_max_ +1)
Z_w=0
S_b= S_w*S_x
Z_b=0

weight_tensor=weight.data.clone()
bias_tensor=bias.data.clone()
quanted_weight=torch.zeros_like(weight_tensor)
quanted_bias=torch.zeros_like(bias_tensor)
weight_dequanted=torch.zeros_like(weight_tensor)
bias_dequanted=torch.zeros_like(bias_tensor)

quanted_weight=torch.round(weight_tensor/S_w+Z_w)
weight_dequanted=(quanted_weight-Z_w)*S_w
quanted_bias=torch.round(bias_tensor/S_b+Z_b)
bias_dequanted=(quanted_bias-Z_b)*S_b


q_bias=quanted_bias - Z_x*quanted_weight.sum(dim=1)
sum=0
for images,_ in test_DL:
    input=input_sub(images.to(DEVICE))
    output=output_sub(input)

    q_x=torch.round(input/S_x+Z_x)
    q_y=torch.matmul(q_x,quanted_weight.T) + q_bias
    q_y=q_y*((S_w*S_x)/S_y) + Z_y
    
    dequanted_y=(q_y - Z_y)*S_y

    sum+=torch.abs(output-dequanted_y).mean().item()


print(f"Average error: {sum/len(test_DL):.4f}")

Average error: 0.0116


In [12]:
bit_num=8
idx=bit_num-2
q_max_=q_max[idx]
q_min_=q_min[idx]

S_X_list=[]
S_Y_list=[]
Z_X_list=[]
Z_Y_list=[]

for images,_ in val_DL:
    input=input_sub(images.to(DEVICE))
    output=output_sub(input)

    r_x_max=input.amax(dim=1)
    r_x_min=input.amin(dim=1)
    S_x= (r_x_max - r_x_min) / (q_max_ - q_min_)
    Z_x=torch.round(q_min_ - r_x_min/S_x)

    r_y_max=output.amax(dim=1)
    r_y_min=output.amin(dim=1)
    S_y= (r_y_max - r_y_min) / (q_max_ - q_min_)
    Z_y=torch.round(q_min_ - r_y_min/S_y)
    

    S_X_list.append(S_x.mean().item())
    S_Y_list.append(S_y.mean().item())
    Z_X_list.append(Z_x.mean().item())
    Z_Y_list.append(Z_y.mean().item())

S_x=np.mean(S_X_list)
Z_x=np.mean(np.round(Z_X_list))
S_y=np.mean(S_Y_list)
Z_y=np.mean(np.round(Z_Y_list))

bias=list(model.classifier.children())[3].bias
weight=list(model.classifier.children())[3].weight

r_w_max=torch.max(torch.abs(weight))
r_b_max=torch.max(torch.abs(bias))
S_w= (r_w_max) / (q_max_ +1)
Z_w=0
S_b= S_w*S_x
Z_b=0

weight_tensor=weight.data.clone()
bias_tensor=bias.data.clone()
quanted_weight=torch.zeros_like(weight_tensor)
quanted_bias=torch.zeros_like(bias_tensor)
weight_dequanted=torch.zeros_like(weight_tensor)
bias_dequanted=torch.zeros_like(bias_tensor)

quanted_weight=torch.round(weight_tensor/S_w+Z_w)
weight_dequanted=(quanted_weight-Z_w)*S_w
quanted_bias=torch.round(bias_tensor/S_b+Z_b)
bias_dequanted=(quanted_bias-Z_b)*S_b


q_bias=quanted_bias - Z_x*quanted_weight.sum(dim=1)

sum=0
for images,_ in test_DL:
    input=input_sub(images.to(DEVICE))
    output=output_sub(input)

    q_x=torch.round(input/S_x+Z_x)
    q_y=torch.matmul(q_x,quanted_weight.T) + q_bias
    q_y=q_y*((S_w*S_x)/S_y) + Z_y
    
    dequanted_y=(q_y - Z_y)*S_y

    sum+=torch.abs(output-dequanted_y).mean().item()


print(f"Average error: {sum/len(test_DL):.4f}")

Average error: 0.0014
